## Classification

To run this script, you need the following files found in the /data directory:
- "final_labels_SG1.xlsx"
- "final_labels_SG2.xlsx"

## Imports and set-up

In [ ]:
# Resolve paths relative to the repository root so this notebook runs from the repo
# root, from notebooks/, or on Colab after cloning the repo:
#   !git clone https://github.com/yasmeenhany/media-bias-detection.git
#   %cd media-bias-detection
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)

In [3]:

!nvidia-smi

Sun Dec  8 15:26:23 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [6]:
tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)
os.environ['PYTHONHASHSEED']=str(0)
os.environ['TF_DETERMINISTIC_OPS'] = '0'

In [7]:
# see if hardware accelerator available
tf.config.experimental.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [8]:
tf.test.gpu_device_name()

'/device:GPU:0'

If GPUs are available, tensorflow will give priority to it automatically and computations will be performed on the GPU as default. That behavior can be changed by assigning a task explicitly to a device. Example:

```
with tf.device('/CPU:0'):
```



## Preprocessing

In [9]:
PATH_sg1 = "data/final_labels_SG1.xlsx"
PATH_sg2 = "data/final_labels_SG2.xlsx"
df_sg1 = pd.read_excel(PATH_sg1)
df_sg2 = pd.read_excel(PATH_sg2)
df_sg1.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg2.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg1.head()

,sentence,news_link,outlet,topic,type,Label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,Biased,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,Non-biased,Somewhat factual but also opinionated,[]
2,Ingraham began the exchange by noting American...,https://www.breitbart.com/economy/2020/01/12/d...,breitbart,immigration,right,No agreement,No agreement,['flood']
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,Biased,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,Non-biased,Entirely factual,[]


In [10]:

# binarize classification problem
df_sg1 = df_sg1[df_sg1['Label_bias']!='No agreement']
df_sg1 = df_sg1[df_sg1['Label_bias'].isna()==False]
df_sg1.replace(to_replace='Biased', value=1, inplace=True)
df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)

df_sg2 = df_sg2[df_sg2['Label_bias']!='No agreement']
df_sg2.replace(to_replace='Biased', value=1, inplace=True)
df_sg2.replace(to_replace='Non-biased', value=0, inplace=True)

<ipython-input-10-3610be668419>:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)
<ipython-input-10-3610be668419>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg2.replace(to_replace='Biased', value=1, inplace=True)
<ipython-input-10-3610be668419>:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss
import numpy as np

def run_naive_bayes_5fold(df_train, max_features=5000, n_splits=5):

    Y = df_train['Label_bias']
    X = df_train['sentence']
    skfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    val_loss = []
    val_acc = []
    val_prec = []
    val_rec = []
    val_f1 = []
    val_f1_micro = []
    val_f1_wmacro = []

    model = MultinomialNB()

    for fold, (train_index, val_index) in enumerate(skfold.split(X, Y), start=1):
        train_texts, val_texts = X.iloc[train_index], X.iloc[val_index]
        train_labels, val_labels = Y.iloc[train_index], Y.iloc[val_index]

        vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
        X_train_tfidf = vectorizer.fit_transform(train_texts)
        X_val_tfidf = vectorizer.transform(val_texts)

        model.fit(X_train_tfidf, train_labels)

        val_probs = model.predict_proba(X_val_tfidf)

        def compute_metrics(labels, preds):

            loss = log_loss(labels, val_probs)
            preds = np.argmax(val_probs, axis=1)
            precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
            f1_micro = precision_recall_fscore_support(labels, preds, average='micro')[2]
            f1_macro = precision_recall_fscore_support(labels, preds, average='macro')[2]
            acc = accuracy_score(labels, preds)

            return {
                'loss': loss,
                'accuracy': acc,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'f1_micro': f1_micro,
                'f1_macro': f1_macro
            }

        eval_result = compute_metrics(val_labels, val_probs)

        val_loss.append(eval_result.get('loss', None))
        val_acc.append(eval_result.get('accuracy', None))
        val_prec.append(eval_result.get('precision', None))
        val_rec.append(eval_result.get('recall', None))
        val_f1.append(eval_result.get('f1', None))
        val_f1_micro.append(eval_result.get('f1_micro', None))
        val_f1_wmacro.append(eval_result.get('f1_macro', None))


    return val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro


In [12]:
val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro = run_naive_bayes_5fold(df_sg1)

loss_cv = np.mean(val_loss)
acc_cv = np.mean(val_acc)
prec_cv = np.mean(val_prec)
rec_cv = np.mean(val_rec)
f1_cv = np.mean(val_f1)

f1_micro_cv = np.mean(val_f1_micro)
f1_wmacro_cv = np.mean(val_f1_wmacro)

print('5-Fold CV Loss: {}'.format(loss_cv))
print('5-Fold CV Accuracy: {}'.format(acc_cv))
print('5-Fold CV Precision: {}'.format(prec_cv))
print('5-Fold CV Recall: {}'.format(rec_cv))
print('5-Fold CV F1 Score: {}'.format(f1_cv))
print('5-Fold CV Micro F1 Score: {}'.format(f1_micro_cv))
print('5-Fold CV Weighted Macro F1 Score: {}'.format(f1_wmacro_cv))

5-Fold CV Loss: 0.6158301365823351
5-Fold CV Accuracy: 0.6791690155548595
5-Fold CV Precision: 0.6805171904986287
5-Fold CV Recall: 0.6791690155548595
5-Fold CV F1 Score: 0.6775079325413695
5-Fold CV Micro F1 Score: 0.6791690155548595
5-Fold CV Weighted Macro F1 Score: 0.6765453097295004


In [13]:
val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro = run_naive_bayes_5fold(df_sg2)

loss_cv = np.mean(val_loss)
acc_cv = np.mean(val_acc)
prec_cv = np.mean(val_prec)
rec_cv = np.mean(val_rec)
f1_cv = np.mean(val_f1)

f1_micro_cv = np.mean(val_f1_micro)
f1_wmacro_cv = np.mean(val_f1_wmacro)

print('5-Fold CV Loss: {}'.format(loss_cv))
print('5-Fold CV Accuracy: {}'.format(acc_cv))
print('5-Fold CV Precision: {}'.format(prec_cv))
print('5-Fold CV Recall: {}'.format(rec_cv))
print('5-Fold CV F1 Score: {}'.format(f1_cv))
print('5-Fold CV Micro F1 Score: {}'.format(f1_micro_cv))
print('5-Fold CV Weighted Macro F1 Score: {}'.format(f1_wmacro_cv))

5-Fold CV Loss: 0.5578136474858693
5-Fold CV Accuracy: 0.7168559194795084
5-Fold CV Precision: 0.7171551450673006
5-Fold CV Recall: 0.7168559194795084
5-Fold CV F1 Score: 0.7167133846823956
5-Fold CV Micro F1 Score: 0.7168559194795084
5-Fold CV Weighted Macro F1 Score: 0.7166282215888718
